# Hibiki-Zero Evaluation — Days 2 to 6
**CSE465 University Project** | Free Colab T4 | moshi 0.2.13

Run the **SETUP CELL** first every single session before anything else.

---
## ⚙️ SETUP — Run this FIRST every session

In [ ]:
# ═══════════════════════════════════════════════════════
# SETUP CELL — Mount Drive + create folders
# Run this FIRST every single session
# ═══════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = "/content/drive/MyDrive/hibiki_project"

folders = [
    PROJECT_ROOT,
    f"{PROJECT_ROOT}/test_audio/casual",
    f"{PROJECT_ROOT}/test_audio/news",
    f"{PROJECT_ROOT}/outputs",
    f"{PROJECT_ROOT}/results",
    f"{PROJECT_ROOT}/logs",
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print('✅ Drive mounted.')
print(f'Project root: {PROJECT_ROOT}')

---
## 📦 Install packages — Run once per session

In [ ]:
# Install all packages needed for Days 2-6
!pip install -q moshi==0.2.13
!pip install -q datasets soundfile scipy
!pip install -q gtts
!pip install -q openai-whisper
!pip install -q sacrebleu
!pip install -q editdistance
!pip install -q scikit-learn matplotlib
!pip install -q huggingface_hub
!apt-get install -q ffmpeg
print('✅ All packages installed')

---
## 📅 DAY 2 — Build Test Set (4 Languages, 2 Domains)

In [ ]:
# ── Day 2, Cell 1: Load 5 casual clips per language from Common Voice 12 ──
from datasets import load_dataset
import soundfile as sf
import json, os

LANGUAGES = {
    'fr': 'French',
    'es': 'Spanish',
    'pt': 'Portuguese',
    'de': 'German'
}
CLIPS_PER_LANG = 5
metadata = []

for lang_code, lang_name in LANGUAGES.items():
    print(f'\n⏳ Loading {lang_name}...')
    ds = load_dataset(
        'mozilla-foundation/common_voice_12_0',
        lang_code,
        split='test',
        streaming=True,
        trust_remote_code=True
    )
    count = 0
    for sample in ds:
        if count >= CLIPS_PER_LANG:
            break
        arr  = sample['audio']['array']
        sr   = sample['audio']['sampling_rate']
        text = sample['sentence']
        dur  = len(arr) / sr
        if dur < 2.0:
            continue
        path = f"{PROJECT_ROOT}/test_audio/casual/{lang_code}_{count+1:02d}.wav"
        sf.write(path, arr, sr)
        metadata.append({
            'file':           path,
            'language':       lang_code,
            'lang_name':      lang_name,
            'domain':         'casual',
            'reference_text': text,
            'reference_en':   None,
            'duration_sec':   round(dur, 2)
        })
        print(f'  [{count+1}/{CLIPS_PER_LANG}] {dur:.1f}s — {text[:60]}...')
        count += 1

print(f'\n✅ Casual clips saved: {len(metadata)}')

In [ ]:
# ── Day 2, Cell 2: Create news-domain clips using gTTS ──
from gtts import gTTS
import soundfile as sf2, os

NEWS = {
    'fr': (
        'Les négociations entre les deux pays ont repris après une pause de plusieurs semaines.',
        'Negotiations between the two countries resumed after a pause of several weeks.'
    ),
    'es': (
        'El gobierno anunció nuevas medidas económicas para combatir la inflación creciente.',
        'The government announced new economic measures to combat rising inflation.'
    ),
    'pt': (
        'O parlamento aprovou hoje uma nova lei sobre proteção de dados pessoais.',
        'Parliament today approved a new law on the protection of personal data.'
    ),
    'de': (
        'Die Bundesregierung hat neue Maßnahmen zur Bekämpfung des Klimawandels angekündigt.',
        'The federal government announced new measures to combat climate change.'
    ),
}

for lang_code, (src_text, ref_en) in NEWS.items():
    lang_name = LANGUAGES[lang_code]
    mp3_tmp   = f'/tmp/{lang_code}_news.mp3'
    wav_final = f"{PROJECT_ROOT}/test_audio/news/{lang_code}_news.wav"

    gTTS(text=src_text, lang=lang_code).save(mp3_tmp)
    os.system(f'ffmpeg -y -i {mp3_tmp} -ar 24000 {wav_final} -loglevel quiet')

    data, sr = sf2.read(wav_final)
    dur = round(len(data) / sr, 2)

    metadata.append({
        'file':           wav_final,
        'language':       lang_code,
        'lang_name':      lang_name,
        'domain':         'news',
        'reference_text': src_text,
        'reference_en':   ref_en,
        'duration_sec':   dur
    })
    print(f'✅ {lang_name} news — {dur}s')

print(f'\nTotal clips: {len(metadata)}')

In [ ]:
# ── Day 2, Cell 3: Verify all files and save metadata to Drive ──
import soundfile as sf3, json

print('=== FILE VERIFICATION ===')
errors = 0
for e in metadata:
    try:
        d, s = sf3.read(e['file'])
        e['duration_sec'] = round(len(d)/s, 2)
        print(f"✅ {e['lang_name']:<12} {e['domain']:<8} {e['duration_sec']}s")
    except Exception as ex:
        print(f"❌ {e['file']} — {ex}")
        errors += 1

meta_path = f"{PROJECT_ROOT}/test_metadata.json"
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'\n✅ Metadata saved → {meta_path}')
print(f'Total: {len(metadata)} clips | Errors: {errors}')
print('\n✅ END OF DAY 2')

---
## 📅 DAY 3 — Run All 24 Clips Through Hibiki-Zero

> ⚠️ Run SETUP CELL + Install Cell first if starting a new session.
> Then re-run Day 2 Cell 3 to reload metadata, OR run the Load cell below.

In [ ]:
# ── Day 3, Cell 1: Load metadata from Drive (run if metadata not in memory) ──
import json

with open(f"{PROJECT_ROOT}/test_metadata.json", 'r', encoding='utf-8') as f:
    metadata = json.load(f)

LANGUAGES = {'fr':'French','es':'Spanish','pt':'Portuguese','de':'German'}
print(f'✅ Loaded {len(metadata)} clips')
for e in metadata:
    print(f"  {e['lang_name']:<12} {e['domain']:<8} {e['file']}")

In [ ]:
# ── Day 3, Cell 2: Run Hibiki-Zero on all 24 clips ──
# --half flag is REQUIRED for T4 (bf16 not supported natively)
import subprocess, time, json, os
import torch
from IPython.display import Audio, display

timing_log = []
failed     = []

for i, entry in enumerate(metadata):
    src  = entry['file']
    base = os.path.splitext(os.path.basename(src))[0]
    out  = f"{PROJECT_ROOT}/outputs/{base}_EN.wav"

    # Skip if already translated — safe to re-run after disconnect
    if os.path.exists(out):
        print(f'[{i+1:02d}/24] ⏭  Already done: {base}')
        entry['output_file'] = out
        continue

    torch.cuda.reset_peak_memory_stats()
    start = time.time()

    result = subprocess.run([
        'python', '-m', 'moshi.run_inference',
        src, out,
        '--hf-repo', 'kyutai/hibiki-zero-3b-pytorch-bf16',
        '--half'   # REQUIRED for T4
    ], capture_output=True, text=True)

    elapsed   = round(time.time() - start, 1)
    peak_vram = round(torch.cuda.max_memory_allocated() / 1e9, 2)

    if os.path.exists(out):
        entry['output_file'] = out
        timing_log.append({
            'clip':          base,
            'language':      entry['language'],
            'domain':        entry['domain'],
            'duration_sec':  entry['duration_sec'],
            'inference_sec': elapsed,
            'peak_vram_gb':  peak_vram,
        })
        print(f'[{i+1:02d}/24] ✅ {entry["lang_name"]} {entry["domain"]} — {elapsed}s | {peak_vram}GB VRAM')
    else:
        failed.append({'clip': base, 'error': result.stderr[-500:]})
        print(f'[{i+1:02d}/24] ❌ FAILED: {base}')
        print(result.stderr[-300:])

# Save timing log and updated metadata
with open(f"{PROJECT_ROOT}/logs/timing_log.json", 'w') as f:
    json.dump(timing_log, f, indent=2)
with open(f"{PROJECT_ROOT}/test_metadata.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'\n✅ Done. {len(timing_log)} translated | {len(failed)} failed')
if failed:
    print('Failed:', [x['clip'] for x in failed])

In [ ]:
# ── Day 3, Cell 3: Listen to one output per language ──
from IPython.display import Audio, display

for lang_code in ['fr','es','pt','de']:
    entry = next((e for e in metadata
                  if e['language'] == lang_code
                  and e['domain'] == 'casual'
                  and 'output_file' in e), None)
    if not entry:
        print(f'No output for {lang_code}, skipping')
        continue
    print(f"\n── {entry['lang_name']} ──")
    print(f"Source text : {entry['reference_text']}")
    print('Input (source language):')
    display(Audio(entry['file']))
    print('Output (English translation):')
    display(Audio(entry['output_file']))

print('\n✅ END OF DAY 3')

---
## 📅 DAY 4 — Transcribe Outputs + Compute BLEU Scores

> ⚠️ Run SETUP CELL + Install Cell + Day 3 Load Cell first if new session.

In [ ]:
# ── Day 4, Cell 1: Load Whisper and transcribe all output audio ──
import whisper, json, os

print('⏳ Loading Whisper small model (~460MB)...')
whisper_model = whisper.load_model('small')
print('✅ Whisper loaded\n')

for i, entry in enumerate(metadata):
    if 'output_file' not in entry:
        print(f'[{i+1:02d}] ⏭  No output file, skipping')
        continue
    if entry.get('transcription_en', ''):
        print(f'[{i+1:02d}] ⏭  Already transcribed')
        continue

    result = whisper_model.transcribe(
        entry['output_file'],
        language='en',
        fp16=True
    )
    transcription = result['text'].strip()
    entry['transcription_en'] = transcription
    print(f"[{i+1:02d}] {entry['lang_name']:<12} {entry['domain']:<8} → {transcription[:70]}...")

with open(f"{PROJECT_ROOT}/test_metadata.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('\n✅ All transcriptions saved to Drive')

In [ ]:
# ── Day 4, Cell 2: Compute BLEU score for news clips ──
import sacrebleu, json

bleu_scores = []

for entry in metadata:
    if entry.get('domain') != 'news':
        continue
    if not entry.get('transcription_en'):
        continue
    if not entry.get('reference_en'):
        continue

    hyp    = entry['transcription_en']
    ref    = entry['reference_en']
    result = sacrebleu.corpus_bleu([hyp], [[ref]])
    score  = round(result.score, 2)

    entry['bleu_score'] = score
    bleu_scores.append({
        'language':   entry['lang_name'],
        'domain':     entry['domain'],
        'reference':  ref,
        'hypothesis': hyp,
        'bleu':       score,
    })
    print(f"{entry['lang_name']:<12} BLEU: {score}")
    print(f"  REF : {ref}")
    print(f"  HYP : {hyp}\n")

with open(f"{PROJECT_ROOT}/results/bleu_scores.json", 'w') as f:
    json.dump(bleu_scores, f, indent=2)
with open(f"{PROJECT_ROOT}/test_metadata.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'✅ BLEU scores saved → {PROJECT_ROOT}/results/bleu_scores.json')

In [ ]:
# ── Day 4, Cell 3: Compute average latency per language and domain ──
import json
from collections import defaultdict

with open(f"{PROJECT_ROOT}/logs/timing_log.json") as f:
    timing = json.load(f)

lang_times = defaultdict(list)
dom_times  = defaultdict(list)

for t in timing:
    lang_times[t['language']].append(t['inference_sec'])
    dom_times[t['domain']].append(t['inference_sec'])

latency_summary = {}
print('=== Average Latency by Language ===')
for lang, times in lang_times.items():
    avg = round(sum(times)/len(times), 1)
    latency_summary[lang] = avg
    print(f'  {lang}: {avg}s avg over {len(times)} clips')

print('\n=== Average Latency by Domain ===')
for dom, times in dom_times.items():
    avg = round(sum(times)/len(times), 1)
    print(f'  {dom}: {avg}s avg over {len(times)} clips')

with open(f"{PROJECT_ROOT}/results/latency_summary.json", 'w') as f:
    json.dump(latency_summary, f, indent=2)

print('\n✅ END OF DAY 4')

---
## 📅 DAY 5 — Results Table + Project Summary

> ⚠️ Run SETUP CELL + Install Cell + Day 3 Load Cell first if new session.

In [ ]:
# ── Day 5, Cell 1: Build main results table ──
import json, csv

with open(f"{PROJECT_ROOT}/results/bleu_scores.json") as f:
    bleu = json.load(f)
with open(f"{PROJECT_ROOT}/results/latency_summary.json") as f:
    latency = json.load(f)

LANG_NAMES = {'fr':'French','es':'Spanish','pt':'Portuguese','de':'German'}
bleu_by_lang = {b['language']: b['bleu'] for b in bleu}

print('=' * 65)
print(f"{'Language':<14} {'BLEU (news)':<16} {'Avg Latency (s)':<20} {'VRAM (GB)'}")
print('-' * 65)

rows = []
for code, name in LANG_NAMES.items():
    bleu_val = bleu_by_lang.get(name, 'N/A')
    lat_val  = latency.get(code, 'N/A')
    print(f"{name:<14} {str(bleu_val):<16} {str(lat_val):<20} see logs")
    rows.append([name, bleu_val, lat_val])

print('=' * 65)

csv_path = f"{PROJECT_ROOT}/results/final_results.csv"
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Language','BLEU_news','Avg_Latency_sec'])
    writer.writerows(rows)

print(f'\n✅ Saved → {csv_path}')

In [ ]:
# ── Day 5, Cell 2: Domain comparison table ──
import json
from collections import defaultdict

domain_summary = defaultdict(list)
for entry in metadata:
    if 'transcription_en' not in entry:
        continue
    words = len(entry['transcription_en'].split())
    domain_summary[entry['domain']].append({
        'lang':  entry['lang_name'],
        'words': words,
        'bleu':  entry.get('bleu_score', None)
    })

print('=== Domain Comparison ===')
for dom, items in domain_summary.items():
    avg_words = round(sum(x['words'] for x in items)/len(items), 1)
    bleu_vals = [x['bleu'] for x in items if x['bleu'] is not None]
    avg_bleu  = round(sum(bleu_vals)/len(bleu_vals), 2) if bleu_vals else 'N/A'
    print(f'  {dom:<10} avg output words: {avg_words}  avg BLEU: {avg_bleu}')

with open(f"{PROJECT_ROOT}/results/domain_summary.json", 'w') as f:
    json.dump(dict(domain_summary), f, indent=2)
print('✅ Domain summary saved')

In [ ]:
# ── Day 5, Cell 3: Print project summary ──
import json

with open(f"{PROJECT_ROOT}/results/bleu_scores.json") as f:
    bleu_data = json.load(f)

best  = max(bleu_data, key=lambda x: x['bleu'])
worst = min(bleu_data, key=lambda x: x['bleu'])

summary = f"""
╔══════════════════════════════════════════════════════════╗
║           PROJECT SUMMARY — CSE465                      ║
╠══════════════════════════════════════════════════════════╣
║ Model   : kyutai/hibiki-zero-3b-pytorch-bf16 + --half   ║
║ GPU     : Tesla T4 (15.36 GB VRAM)                      ║
╠══════════════════════════════════════════════════════════╣
║ Test set: 20 casual + 4 news clips (4 EU languages)     ║
║ Best BLEU  : {best['language']:<10} {best['bleu']:.2f}                          ║
║ Worst BLEU : {worst['language']:<10} {worst['bleu']:.2f}                          ║
╠══════════════════════════════════════════════════════════╣
║ Original contribution vs Ohashi et al. 2025:            ║
║  1. Compared 4 EU languages (paper: Japanese only)      ║
║  2. Tested news domain (paper: casual/travel only)      ║
║  3. Measured latency on T4 (paper: H100/V100 only)      ║
╚══════════════════════════════════════════════════════════╝"""

print(summary)
with open(f"{PROJECT_ROOT}/results/project_summary.txt", 'w') as f:
    f.write(summary)

print('\n✅ END OF DAY 5')

---
## 📅 DAY 6 — Classical Pattern Recognition Extension

### Task 1: K-Means Clustering (Syllabus: cluster seeking)
### Task 2: LDA Classification (Syllabus: linear decision functions)
### Task 3: Levenshtein Distance (Syllabus: string-to-string distance)

> ⚠️ Run SETUP CELL + Install Cell + Day 3 Load Cell first if new session.

In [ ]:
# ── Day 6, Task 1 Cell 1: Extract Mimi embeddings for all 24 clips ──
# Uses the confirmed moshi 0.2.13 API:
# hf_hub_download to get weight path, then loaders.get_mimi(path, device)

from huggingface_hub import hf_hub_download
from moshi.models import loaders
import torch, json, os, numpy as np
import torchaudio

print('⏳ Downloading Mimi weights...')
mimi_weight = hf_hub_download(
    repo_id='kyutai/hibiki-zero-3b-pytorch-bf16',
    filename=loaders.MIMI_NAME
)
print(f'✅ Mimi weight at: {mimi_weight}')

mimi = loaders.get_mimi(mimi_weight, device='cuda')
mimi.set_num_codebooks(8)
mimi.eval()
print('✅ Mimi loaded on GPU\n')

embeddings    = []
labels_lang   = []
labels_domain = []

for entry in metadata:
    if not os.path.exists(entry['file']):
        continue

    # Load and resample to 24kHz
    wav, sr = torchaudio.load(entry['file'])
    if sr != 24000:
        wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.mean(0, keepdim=True).unsqueeze(0).to('cuda')  # (1,1,T)

    with torch.no_grad():
        codes = mimi.encode(wav)                     # (1, 8, T)
        feat  = codes[0].float().mean(dim=1).cpu().numpy()  # mean over time → (8,)

    embeddings.append(feat)
    labels_lang.append(entry['language'])
    labels_domain.append(entry['domain'])
    print(f"✅ {entry['lang_name']:<12} {entry['domain']:<8} feat={np.round(feat,2)}")

X = np.array(embeddings)  # shape (24, 8)
print(f'\nFeature matrix shape: {X.shape}')

np.save(f"{PROJECT_ROOT}/results/mimi_embeddings.npy", X)
with open(f"{PROJECT_ROOT}/results/embedding_labels.json", 'w') as f:
    json.dump({'lang': labels_lang, 'domain': labels_domain}, f)

print('✅ Embeddings saved to Drive')

In [ ]:
# ── Day 6, Task 1 Cell 2: K-Means clustering + PCA plot ──
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np, json

X           = np.load(f"{PROJECT_ROOT}/results/mimi_embeddings.npy")
with open(f"{PROJECT_ROOT}/results/embedding_labels.json") as f:
    lbls = json.load(f)
labels_lang   = lbls['lang']
labels_domain = lbls['domain']

def cluster_purity(cluster_labels, true_labels):
    from collections import Counter
    total = 0
    for c in set(cluster_labels):
        idx    = [i for i, cl in enumerate(cluster_labels) if cl == c]
        counts = Counter([true_labels[i] for i in idx])
        total += max(counts.values())
    return total / len(cluster_labels)

results = {}
for k in [2, 4]:
    km   = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    pred = km.labels_.tolist()
    pl   = cluster_purity(pred, labels_lang)
    pd_  = cluster_purity(pred, labels_domain)
    results[f'k={k}'] = {
        'purity_by_language': round(pl, 3),
        'purity_by_domain':   round(pd_, 3),
        'assignments':        pred
    }
    print(f'k={k}  purity(language)={pl:.3f}  purity(domain)={pd_:.3f}')

# PCA scatter
pca  = PCA(n_components=2)
X2d  = pca.fit_transform(X)
km4  = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X)

LANG_COLORS = {'fr':'#2d7dd2','es':'#e84855','pt':'#3bb273','de':'#f7a325'}
DOM_MARKERS = {'casual':'o','news':'^'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('PCA of Mimi Embeddings', fontsize=12)

for i, (x, y) in enumerate(X2d):
    axes[0].scatter(x, y, color=LANG_COLORS[labels_lang[i]],
                    marker=DOM_MARKERS[labels_domain[i]],
                    s=80, edgecolors='k', linewidths=0.5)
axes[0].set_title('True Labels (colour=language, shape=domain)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

CLUSTER_COLORS = ['#7b2d8b','#e84855','#3bb273','#f7a325']
for i, (x, y) in enumerate(X2d):
    axes[1].scatter(x, y, color=CLUSTER_COLORS[km4.labels_[i]],
                    marker=DOM_MARKERS[labels_domain[i]],
                    s=80, edgecolors='k', linewidths=0.5)
axes[1].set_title('K-Means Clusters (k=4)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.tight_layout()
plot_path = f"{PROJECT_ROOT}/results/task1_kmeans_pca.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

with open(f"{PROJECT_ROOT}/results/task1_kmeans_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f'✅ Plot saved  → {plot_path}')
print(f'✅ Results saved')

In [ ]:
# ── Day 6, Task 2: LDA language classification with LOOCV ──
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np, matplotlib.pyplot as plt, json

X           = np.load(f"{PROJECT_ROOT}/results/mimi_embeddings.npy")
with open(f"{PROJECT_ROOT}/results/embedding_labels.json") as f:
    lbls = json.load(f)
labels_lang = lbls['lang']
labels_domain = lbls['domain']

LANG_NAMES = {'fr':'French','es':'Spanish','pt':'Portuguese','de':'German'}
classes    = ['fr','es','pt','de']
y          = np.array(labels_lang)

# Leave-One-Out Cross Validation
loo   = LeaveOneOut()
preds = []
for train_idx, test_idx in loo.split(X):
    clf = LDA()
    clf.fit(X[train_idx], y[train_idx])
    preds.append(clf.predict(X[test_idx])[0])

preds = np.array(preds)
acc   = (preds == y).mean()
cm    = confusion_matrix(y, preds, labels=classes)

print(f'LDA LOOCV Accuracy: {acc:.3f} ({int(acc*len(y))}/{len(y)} correct)')
print('\nConfusion matrix (rows=true, cols=predicted):')
print('      fr   es   pt   de')
for i, row in enumerate(cm):
    print(f'  {classes[i]}  {row}')

# Confusion matrix plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[LANG_NAMES[c] for c in classes]
)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'LDA Confusion Matrix (LOOCV Acc = {acc:.2f})')

# LDA 2D projection
clf_full = LDA(n_components=2)
X_lda    = clf_full.fit_transform(X, y)
LANG_COLORS = {'fr':'#2d7dd2','es':'#e84855','pt':'#3bb273','de':'#f7a325'}
for i, (x, y2) in enumerate(X_lda):
    axes[1].scatter(x, y2, color=LANG_COLORS[labels_lang[i]],
                    s=90, edgecolors='k', linewidths=0.6)
from matplotlib.patches import Patch
handles = [Patch(color=LANG_COLORS[c], label=LANG_NAMES[c]) for c in classes]
axes[1].legend(handles=handles)
axes[1].set_xlabel('LD1'); axes[1].set_ylabel('LD2')
axes[1].set_title('LDA Discriminant Space')

plt.tight_layout()
lda_path = f"{PROJECT_ROOT}/results/task2_lda_plots.png"
plt.savefig(lda_path, dpi=150, bbox_inches='tight')
plt.show()

lda_results = {
    'loocv_accuracy':   round(float(acc), 3),
    'n_samples':        len(y),
    'confusion_matrix': cm.tolist(),
    'classes':          classes,
    'predictions':      preds.tolist(),
    'true_labels':      y.tolist(),
}
with open(f"{PROJECT_ROOT}/results/task2_lda_results.json", 'w') as f:
    json.dump(lda_results, f, indent=2)

print(f'\n✅ LDA plots saved → {lda_path}')

In [ ]:
# ── Day 6, Task 3: Levenshtein / Edit Distance on Transcriptions ──
import editdistance, json, os
import numpy as np
import matplotlib.pyplot as plt

def wer(ref, hyp):
    r = ref.lower().split()
    h = hyp.lower().split()
    return editdistance.eval(r, h) / max(len(r), 1)

def cer(ref, hyp):
    r = list(ref.lower().replace(' ',''))
    h = list(hyp.lower().replace(' ',''))
    return editdistance.eval(r, h) / max(len(r), 1)

edit_results = []

for entry in metadata:
    hyp = entry.get('transcription_en','').strip()
    ref = entry.get('reference_en','').strip()
    if not hyp or not ref:
        continue
    w    = round(wer(ref, hyp), 3)
    c    = round(cer(ref, hyp), 3)
    dist = editdistance.eval(ref.lower().split(), hyp.lower().split())
    edit_results.append({
        'lang_name':  entry['lang_name'],
        'language':   entry['language'],
        'domain':     entry['domain'],
        'wer':        w,
        'cer':        c,
        'edit_dist':  dist,
        'reference':  ref,
        'hypothesis': hyp,
    })
    print(f"{entry['lang_name']:<12} {entry['domain']:<8} WER={w:.3f}  CER={c:.3f}  edit={dist}")
    print(f"  REF: {ref[:70]}")
    print(f"  HYP: {hyp[:70]}\n")

with open(f"{PROJECT_ROOT}/results/task3_edit_distance.json", 'w') as f:
    json.dump(edit_results, f, ensure_ascii=False, indent=2)
print('✅ Edit distance results saved')

In [ ]:
# ── Day 6, Task 3 Cell 2: Alignment visualisation (DP traceback) ──
import matplotlib.pyplot as plt
import json

with open(f"{PROJECT_ROOT}/results/task3_edit_distance.json") as f:
    edit_results = json.load(f)

def align_words(ref_words, hyp_words):
    r, h = ref_words, hyp_words
    n, m = len(r), len(h)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            if r[i-1] == h[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    ops = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and r[i-1] == h[j-1]:
            ops.append(('MATCH', r[i-1], h[j-1])); i-=1; j-=1
        elif i > 0 and j > 0 and dp[i][j]==dp[i-1][j-1]+1:
            ops.append(('SUB',   r[i-1], h[j-1])); i-=1; j-=1
        elif i > 0 and dp[i][j]==dp[i-1][j]+1:
            ops.append(('DEL',   r[i-1], '-'));     i-=1
        else:
            ops.append(('INS',   '-',    h[j-1]));  j-=1
    return list(reversed(ops))

OP_COLORS = {'MATCH':'#3bb273','SUB':'#f7a325','INS':'#2d7dd2','DEL':'#e84855'}

n_plots = len(edit_results)
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 3*n_plots))
if n_plots == 1:
    axes = [axes]

fig.suptitle('Word-Level Alignment: Reference vs Hypothesis (news domain)', fontsize=12)

for ax, res in zip(axes, edit_results):
    ref_w = res['reference'].lower().split()
    hyp_w = res['hypothesis'].lower().split()
    ops   = align_words(ref_w, hyp_w)

    ax.set_title(
        f"{res['lang_name']} → EN  (WER={res['wer']:.2f}, edit_dist={res['edit_dist']})",
        fontsize=10, loc='left'
    )
    ax.set_xlim(0, len(ops)+1)
    ax.set_ylim(0, 2.5)
    ax.axis('off')

    for idx, (op, rw, hw) in enumerate(ops):
        x = idx + 0.5
        ax.text(x, 1.8, rw, ha='center', fontsize=8, color='#333')
        ax.text(x, 0.8, hw, ha='center', fontsize=8,
                color=OP_COLORS[op], fontweight='bold')
        ax.text(x, 1.3, op[0], ha='center', fontsize=7, color=OP_COLORS[op])

    ax.text(-0.2, 1.8, 'REF:', ha='right', fontsize=8, fontweight='bold')
    ax.text(-0.2, 0.8, 'HYP:', ha='right', fontsize=8, fontweight='bold')

from matplotlib.patches import Patch
legend = [Patch(color=c, label=op) for op, c in OP_COLORS.items()]
fig.legend(handles=legend, loc='lower right', ncol=4, fontsize=9)

plt.tight_layout()
align_path = f"{PROJECT_ROOT}/results/task3_alignment.png"
plt.savefig(align_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Alignment plot saved → {align_path}')
print('\n✅ END OF DAY 6 — ALL TASKS COMPLETE')